# 05 — Développement de la solution

Ce notebook présente le modèle principal, ses paramètres, ses hypothèses et les justifications de son choix. Le prétraitement vient de l’[étape 4](04_pretraitement.ipynb); l’évaluation détaillée relève de l’étape 6.

**Modèle retenu : la moyenne simple de cinq modèles** (Lasso, ElasticNet, Kernel Ridge, Gradient Boosting et XGBoost), entraînés sur le prétraitement de l’étape 4. Le code est dans `src/training.py`.

## 1. Démarche : trois approches testées

Lors des expériences (`notebooks/experiences/`), les trois approches ont été comparées sur une même partition (80 % des ventes pour l’entraînement, 20 % pour la validation, `random_state=42`) et avec la même métrique : la RMSE des logarithmes des prix, celle de Kaggle. Les scores de cette section viennent de ces expériences. La suite du notebook n’utilise plus de partition : elle évalue le modèle par validation croisée sur toutes les ventes (section 5).

### 1.1 RAG : estimer à partir de ventes comparables

Dans un autre projet portant sur des maisons à vendre sur Centris, un RAG (*Retrieval-Augmented Generation*) donnait de bonnes estimations. Nous avons donc commencé par cette approche. Elle imite un évaluateur immobilier en trois temps :

1. **Indexation.** Chaque maison d’entraînement est décrite par un court texte. Ce texte est converti en vecteur (*embedding*) et stocké dans une base vectorielle, avec le prix de la maison.
2. **Recherche.** Pour une maison à estimer, nous retrouvons les cinq ventes dont la description est la plus proche, de préférence dans le même quartier.
3. **Génération.** Un grand modèle de langage (LLM) reçoit ces cinq comparables, leur prix et la description de la maison, puis estime son prix.

La meilleure version obtient une RMSE log de 0,133.

### 1.2 XGBoost : un modèle conçu pour les données tabulaires

Les données d’Ames sont tabulaires : 79 colonnes de surfaces, de qualités, de codes et d’années. Les modèles de *gradient boosting* sur arbres de décision, comme XGBoost, fonctionnent généralement bien sur ce type de données. Ils captent les seuils et les interactions sans hypothèse de forme. Après réglage des hyperparamètres, XGBoost obtient une RMSE log de 0,125 à 0,130 selon la configuration : il fait mieux que le RAG.

**Pourquoi le RAG fait-il moins bien ici?** Dans l’autre projet, le RAG exploitait surtout des données textuelles qualitatives, comme la description rédigée de chaque maison. Le jeu de données de la compétition n’en contient pas : notre texte était généré à partir des colonnes. Il n’apportait donc aucune information qu’un modèle tabulaire ne lit pas directement, et le LLM pondérait les caractéristiques moins finement qu’un modèle entraîné sur les 1 168 ventes.

### 1.3 Ensemble de modèles : l’architecture retenue

Nous avons enfin combiné cinq modèles de deux familles : trois modèles linéaires et deux modèles d’arbres. En validation croisée, leur moyenne simple fait mieux que chacun d’eux; en validation, elle fait mieux que XGBoost seul. C’est l’architecture retenue, détaillée dans la suite.

| Approche | RMSE log (validation) |
|---|---|
| RAG + LLM, version 1 | 0,140 |
| RAG + LLM, version 2 | 0,133 |
| XGBoost réglé | 0,125 à 0,130 |
| **Ensemble de cinq modèles (expérience 04)** | **0,123** |

Les scores du RAG et de XGBoost portent sur 291 des 292 maisons de validation; celui de l’ensemble porte sur les 292.

## 2. Modèle retenu

```text
Maison brute (79 colonnes)
   │  make_base() — prétraitement de l’étape 4, appris sur l’entraînement
   ▼
227 colonnes numériques
   ├── Lasso ─────────────┐
   ├── ElasticNet ────────┤
   ├── Kernel Ridge ──────┼──► moyenne des log(prix) ──► exp ──► prix prédit
   ├── Gradient Boosting ─┤
   └── XGBoost ───────────┘
```

- **Prétraitement.** La recette de base de l’étape 4 enchaîne nettoyage, valeurs manquantes, 14 variables créées, encodage ordinal et one-hot, `log1p` des longues queues et standardisation. Elle est apprise sur les maisons d’entraînement seulement.
- **Cible.** Chaque modèle apprend `log(SalePrice)`. `TransformedTargetRegressor` applique le logarithme à l’entraînement et l’exponentielle à la prédiction.
- **Combinaison.** `VotingRegressor` fait la moyenne des cinq prédictions sur l’échelle logarithmique, celle de la métrique. Sur l’échelle des prix, c’est une moyenne géométrique.
- **Ventes aberrantes.** Les maisons 524 et 1299 (plus de 4 000 pi², vendues moins de 300 000 $) sont retirées de l’entraînement (`fit_model`), jamais des maisons évaluées ni de `test.csv`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.model_selection import KFold

from src.evaluation import cv_rmse
from src.preprocessing.steps.cleaning import unusual_sales
from src.training import (
    MEMBERS, RANDOM_STATE, cross_validate_model, fit_model, load_data, make_model,
)


def show_title(title):
    display(Markdown(f"**{title}**"))


root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
X, y, test = load_data(root / "data/train.csv", root / "data/test.csv")
print(f"{len(X)} ventes étiquetées dans train.csv, {len(test)} maisons dans test.csv.")

## 3. Paramètres utilisés

| Modèle | Famille | Paramètres | Rôle dans l’ensemble |
|---|---|---|---|
| Lasso | Linéaire, pénalité L1 | Pénalité choisie par validation croisée interne parmi 40 valeurs | Le log du prix est presque une somme d’effets; la pénalité L1 met à zéro les colonnes inutiles |
| ElasticNet | Linéaire, pénalités L1 et L2 | Pénalité et part L1/L2 choisies par validation croisée interne | Partage le poids entre colonnes corrélées au lieu d’en choisir une |
| Kernel Ridge | Linéaire dans un espace élargi | Noyau polynomial de degré 2, `alpha` = 0,6, `coef0` = 2,5 | Ajoute implicitement les carrés et les produits de colonnes |
| Gradient Boosting | Arbres | 3 000 arbres de profondeur 4, taux 0,05, perte de Huber, 15 maisons par feuille au minimum | Seuils et interactions; la perte de Huber atténue les prix extrêmes |
| XGBoost | Arbres | 2 000 arbres de profondeur 4, taux 0,02, 70 % des maisons et 50 % des colonnes par arbre | Configuration réglée lors du test de XGBoost (section 1.2) |

Les pénalités du Lasso et d’ElasticNet sont réglées automatiquement. Les paramètres de Kernel Ridge et de Gradient Boosting sont des valeurs usuelles pour ce jeu de données, sans recherche. Pour XGBoost, une recherche aléatoire n’avait gagné que 0,0006 sur la configuration manuelle : dans cette région, la performance dépend peu des hyperparamètres.

## 4. Hypothèses retenues

1. **Échelle logarithmique.** Le prix est asymétrique (étape 3), et la métrique de Kaggle mesure des écarts relatifs. Modéliser `log(SalePrice)` rend les erreurs comparables entre maisons bon marché et chères.
2. **Ventes typiques.** Les maisons 524 et 1299 sont considérées hors de la population à prédire, comme le recommande l’auteur du jeu de données. Elles sont retirées de l’entraînement, et les scores de validation croisée sont calculés sans elles (`cv_rmse`, étape 4).
3. **Erreurs complémentaires.** Les modèles linéaires et les arbres ne se trompent pas sur les mêmes maisons. En faire la moyenne compense une partie de leurs erreurs (vérifié à la section 5).
4. **Stabilité du marché.** Les ventes de 2006 à 2010 à Ames, y compris pendant la crise de 2008, représentent les ventes de `test.csv`, qui couvrent la même période.

## 5. Justifications techniques

Nous mesurons chaque modèle et leur moyenne par validation croisée à cinq plis sur les 1 460 ventes de `train.csv` (`cross_validate_model`). Le modèle complet, prétraitement compris, est réajusté dans chaque pli, sans les deux ventes aberrantes. Nous ne réservons pas de partition de validation : la validation croisée donne déjà une prédiction hors pli pour chaque vente, et toutes les ventes servent ainsi à l’entraînement comme à l’évaluation.

In [ ]:
# Entrainer les modèles
folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof, _ = cross_validate_model(make_model(), X, y, folds)

In [ ]:
show_title("5.1 Validation croisée des cinq modèles et de leur moyenne")
fold_scores = pd.DataFrame({
    name: [cv_rmse(y.iloc[v], oof[name].iloc[v], X.iloc[v]) for _, v in folds.split(X)]
    for name in oof
})
summary = pd.DataFrame({"RMSE log (moyenne des plis)": fold_scores.mean(),
                        "Écart type entre plis": fold_scores.std(ddof=0)})
summary = summary.rename(index={"Ensemble": "Moyenne des cinq (retenu)"}).sort_values("RMSE log (moyenne des plis)")
display(summary.round(4))

show_title("5.2 RMSE log de chaque pli")
per_fold = (fold_scores.set_axis([f"Pli {k}" for k in range(1, folds.n_splits + 1)]).T
            .rename(index={"Ensemble": "Moyenne des cinq (retenu)"}).loc[summary.index])
display(per_fold.round(4))
wins = per_fold.idxmin().eq("Moyenne des cinq (retenu)").sum()
print(f"La moyenne des cinq obtient la plus faible erreur dans {wins} plis sur {folds.n_splits}.")

show_title("5.3 Corrélation des erreurs hors pli (log, ventes typiques)")
errors = np.log(oof[list(MEMBERS)]).sub(np.log(y), axis=0)
correlation = errors[~unusual_sales(X, y)].corr()
figure, axis = plt.subplots(figsize=(6, 4.8))
image = axis.imshow(correlation, cmap="Blues", vmin=0.5, vmax=1)
axis.set_xticks(range(len(correlation)), correlation.columns, rotation=30, ha="right")
axis.set_yticks(range(len(correlation)), correlation.index)
for i in range(len(correlation)):
    for j in range(len(correlation)):
        value = correlation.iat[i, j]
        axis.text(j, i, f"{value:.2f}".replace(".", ","), ha="center", va="center",
                  color="white" if value > 0.9 else "black", fontsize=9)
figure.colorbar(image, ax=axis, label="Corrélation")
plt.tight_layout()
plt.show()

**Lecture.**
- **La moyenne fait mieux que chaque modèle seul** : 0,1074 contre 0,1100 pour le meilleur, Kernel Ridge, soit 2,4 % d’erreur en moins. Gradient Boosting seul est le moins bon (0,1138). Les écarts sont plus petits que l’écart type entre plis (de 0,007 à 0,010), mais ils sont mesurés sur les mêmes plis : la moyenne obtient la plus faible erreur dans chacun des cinq plis (sortie 5.2).
- **Deux familles d’erreurs** (sortie 5.3). Les trois modèles linéaires se trompent presque de la même façon, et les deux modèles d’arbres aussi. Entre les deux familles, la corrélation baisse nettement : un modèle linéaire et un modèle d’arbres se trompent souvent sur des maisons différentes. C’est là que la moyenne gagne.
- **Moyenne simple plutôt que pondérée.** Une moyenne pondérée, avec des poids appris sur les prédictions hors pli, n’avait rien apporté lors du test de l’ensemble (0,1068 contre 0,1067) : cinq poids appris sur environ 1 160 maisons ajoutent surtout du bruit. La moyenne simple n’a aucun paramètre à apprendre.

## 6. Entraînement final

Le modèle est entraîné sur les 1 460 ventes de `train.csv`, sans les deux ventes aberrantes, pour prédire les 1 459 maisons de `test.csv`.

In [ ]:
show_title("6.1 Soumission Kaggle")
final_model = fit_model(make_model(), X, y)
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": final_model.predict(test)})
assert len(submission) == len(test) and submission["SalePrice"].gt(0).all()
submission_path = root / "submissions" / "05_ensemble.csv"
submission_path.parent.mkdir(exist_ok=True)
submission.to_csv(submission_path, index=False)
print(f"{submission_path.relative_to(root)} : {len(submission)} prix, "
      f"médiane {submission['SalePrice'].median():,.0f} $".replace(",", " "))

## 7. Bilan

- **Démarche.** Le RAG, efficace sur des annonces décrites par du texte, convient mal à des données purement tabulaires (0,133). XGBoost, conçu pour ce type de données, fait mieux (0,125 à 0,130). La moyenne de cinq modèles de deux familles fait mieux encore : 0,123 en validation lors de l’expérience 04, et 0,1074 en validation croisée sur les 1 460 ventes.
- **Modèle retenu.** `make_model()` dans `src/training.py` : prétraitement de l’étape 4 (`make_base()`), puis moyenne des log(prix) prédits par Lasso, ElasticNet, Kernel Ridge, Gradient Boosting et XGBoost.
- **Limites.** Kernel Ridge et Gradient Boosting gardent des paramètres usuels, sans recherche. Lasso et ElasticNet sont presque le même modèle : la moyenne donne en pratique un poids double à cette approche. Les écarts entre modèles (quelques millièmes) sont plus petits que l’écart type entre plis. Sans partition réservée, les scores de validation croisée servent à la fois à choisir et à mesurer : ils sont légèrement optimistes, et seule la soumission Kaggle donne une mesure indépendante. L’étape 6 évaluera le modèle plus en détail.